# Upload FineSightBench-Large to Hugging Face

This notebook packages both splits — **Perception** (42 000 images) and **Reasoning** (39 200 images) — into a single `datasets.DatasetDict` and pushes it to the Hugging Face Hub as **FineSightBench-Large** (a 10× scaled companion to [FineSightBench](https://huggingface.co/datasets/Volavion/FineSightBench)).

| Split | #Samples | Source |
|-------|----------|--------|
| `perception` | 42 000 | `data/full_perception_large/` |
| `reasoning` | 39 200 | `data/full_reasoning_large/` |

FineSightBench now includes a **SynthText-style "text in the wild"** task family (7 000 perception + 14 000 reasoning samples) in addition to the original synthetic-canvas tasks. Text samples render English words onto real natural-scene photographs (from the [SynthText](https://github.com/ankush-me/SynthText) `bg_img` set) with pixel-accurate control over character cap-height.

**Prerequisites**

- Run `huggingface-cli login` once (or set `HF_TOKEN` env var) before executing this notebook.
- Set `HF_REPO_ID` in the *Configuration* cell to your desired `<username>/<dataset-name>`.

## 1. Install / Import

In [1]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import datasets
    import huggingface_hub
    print(f"datasets {datasets.__version__}, huggingface_hub {huggingface_hub.__version__}")
except ImportError:
    _pip("datasets", "huggingface_hub")
    import datasets, huggingface_hub
    print("Installed and imported.")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
/home/snt/projects_lujun/FineSightBench/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Installed and imported.


In [2]:
import json
import os
from pathlib import Path

import finesightbench as _fb
from datasets import Dataset, DatasetDict, Features, Value, Image as HFImage
from PIL import Image

REPO_ROOT = Path(_fb.__file__).resolve().parent.parent
print("Repo root:", REPO_ROOT)

Repo root: /home/snt/projects_lujun/FineSightBench


## 2. Configuration

In [10]:
# ── Edit this cell ────────────────────────────────────────────────────────────
HF_REPO_ID   = "Volavion/FineSightBench-Large"   # HF username/repo
PRIVATE      = False                        # set True to create a private dataset
# ──────────────────────────────────────────────────────────────────────────────

PERCEPTION_DIR = REPO_ROOT / "data" / "full_perception_large"
REASONING_DIR  = REPO_ROOT / "data" / "full_reasoning_large"

print(f"HF repo      : {HF_REPO_ID}")
print(f"Perception   : {PERCEPTION_DIR}")
print(f"Reasoning    : {REASONING_DIR}")

HF repo      : Volavion/FineSightBench
Perception   : /home/snt/projects_lujun/FineSightBench/data/full_perception
Reasoning    : /home/snt/projects_lujun/FineSightBench/data/full_reasoning


## 3. Load Labels

In [4]:
def load_jsonl(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

perception_samples = load_jsonl(PERCEPTION_DIR / "labels.jsonl")
reasoning_samples  = load_jsonl(REASONING_DIR  / "labels.jsonl")

print(f"Perception samples : {len(perception_samples)}")
print(f"Reasoning  samples : {len(reasoning_samples)}")
print()
print("Perception keys:", list(perception_samples[0].keys()))
print("Reasoning  keys:", list(reasoning_samples[0].keys()))

Perception samples : 3500
Reasoning  samples : 2520

Perception keys: ['image_id', 'image_path', 'dataset', 'task_type', 'question', 'answer', 'difficulty', 'generation_mode', 'metadata', 'answer_format']
Reasoning  keys: ['image_id', 'image_path', 'dataset', 'task_type', 'question', 'answer', 'difficulty', 'generation_mode', 'metadata', 'answer_format']


## 4. Build HuggingFace Datasets

Each row contains:
- `image` – PIL image (embedded as bytes in the Arrow table)
- `image_id`, `task_type`, `question`, `answer`, `difficulty`
- `metadata` – JSON string of the full metadata dict

In [5]:
from tqdm.auto import tqdm

def build_rows(samples: list[dict], base_dir: Path) -> list[dict]:
    rows = []
    for s in tqdm(samples, desc=str(base_dir.name)):
        img_path = base_dir / s["image_path"]
        # answer may be dict (reasoning jsonl) or str (perception jsonl)
        answer = s["answer"]
        if isinstance(answer, dict):
            answer = json.dumps(answer, ensure_ascii=False)
        rows.append({
            "image"      : str(img_path),   # will be converted to HF Image later
            "image_id"   : s["image_id"],
            "task_type"  : s["task_type"],
            "question"   : s["question"],
            "answer"     : answer,
            "difficulty" : s.get("difficulty", ""),
            "metadata"   : json.dumps(s.get("metadata", {}), ensure_ascii=False),
        })
    return rows

p_rows = build_rows(perception_samples, PERCEPTION_DIR)
r_rows = build_rows(reasoning_samples,  REASONING_DIR)

print(f"\nPerception rows : {len(p_rows)}")
print(f"Reasoning rows  : {len(r_rows)}")

full_reasoning: 100%|██████████| 2520/2520 [00:00<00:00, 34541.33it/s]


Perception rows : 3500
Reasoning rows  : 2520


In [6]:
# Cast to HF Dataset with Image feature
from datasets import Dataset, Features, Value, Image as HFImage

FEATURES = Features({
    "image"     : HFImage(),
    "image_id"  : Value("string"),
    "task_type" : Value("string"),
    "question"  : Value("string"),
    "answer"    : Value("string"),
    "difficulty": Value("string"),
    "metadata"  : Value("string"),
})

ds_perception = Dataset.from_list(p_rows, features=FEATURES)
ds_reasoning  = Dataset.from_list(r_rows, features=FEATURES)

print(ds_perception)
print(ds_reasoning)

Dataset({
    features: ['image', 'image_id', 'task_type', 'question', 'answer', 'difficulty', 'metadata'],
    num_rows: 3500
})
Dataset({
    features: ['image', 'image_id', 'task_type', 'question', 'answer', 'difficulty', 'metadata'],
    num_rows: 2520
})


In [7]:
dataset_dict = DatasetDict({
    "perception": ds_perception,
    "reasoning" : ds_reasoning,
})
print(dataset_dict)

DatasetDict({
    perception: Dataset({
        features: ['image', 'image_id', 'task_type', 'question', 'answer', 'difficulty', 'metadata'],
        num_rows: 3500
    })
    reasoning: Dataset({
        features: ['image', 'image_id', 'task_type', 'question', 'answer', 'difficulty', 'metadata'],
        num_rows: 2520
    })
})


## 5. Authenticate with Hugging Face

If `HF_TOKEN` is already set as an environment variable this cell is a no-op.  
Otherwise you will be prompted to paste your token.

In [ ]:
from huggingface_hub import HfApi, login

# Use token directly (set via env var HF_TOKEN or hardcoded for this session)
token = os.environ.get("HF_TOKEN", "")
login(token=token, add_to_git_credential=False)

api = HfApi()
user = api.whoami()
print(f"Authenticated as: {user['name']}")

Authenticated as: Volavion


## 6. Push to Hub

In [11]:
print(f"Pushing DatasetDict to: {HF_REPO_ID}")
print(f"  perception : {len(ds_perception)} samples")
print(f"  reasoning  : {len(ds_reasoning)} samples")
print()

dataset_dict.push_to_hub(
    HF_REPO_ID,
    private=PRIVATE,
    num_shards={"perception": 32, "reasoning": 32},
    commit_message="Upload FineSightBench-Large v1.0 (perception 42000 + reasoning 39200, 10× FineSightBench)",
)

print(f"\nDone! Dataset available at: https://huggingface.co/datasets/{HF_REPO_ID}")

Pushing DatasetDict to: Volavion/FineSightBench
  perception : 3500 samples
  reasoning  : 2520 samples



Setting num_proc from 1 back to 1 for the perception split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 76.74ba/s]
Processing Files (1 / 1): 100%|██████████| 6.41MB / 6.41MB, 3.56MB/s  
New Data Upload: 100%|██████████| 6.41MB / 6.41MB, 3.56MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.75s/ shards]
Setting num_proc from 1 back to 1 for the reasoning split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 13.83ba/s]
Processing Files (1 / 1): 100%|██████████| 49.0MB / 49.0MB, 13.6MB/s  
New Data Upload: 100%|██████████| 49.0MB / 49.0MB, 13.6MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:04<00:00,  4.59s/ shards]



Done! Dataset available at: https://huggingface.co/datasets/Volavion/FineSightBench


## 7. Upload README (Dataset Card)

Writes a `README.md` with YAML front-matter so the dataset shows correct metadata on the Hub.

In [ ]:
from huggingface_hub import upload_file

README = """\
---
language:
  - en
license: apache-2.0
task_categories:
  - visual-question-answering
  - image-classification
  - image-to-text
pretty_name: FineSightBench-Large
size_categories:
  - 10K<n<100K
tags:
  - VLM-evaluation
  - fine-grained-visual-perception
  - fine-grained-visual-reasoning
  - text-in-the-wild
  - scene-text-recognition
splits:
  - name: perception
    num_examples: 42000
  - name: reasoning
    num_examples: 39200
---

# FineSightBench-Large

**FineSightBench-Large** is a **10× scaled** edition of [FineSightBench](https://huggingface.co/datasets/Volavion/FineSightBench) — identical task design, difficulty sweep, answer schemas, and image regimes, with every base sample count multiplied by ten for higher statistical power and robust per-(task, size, count) evaluation.

**FineSightBench** is a fine-grained visual benchmark for evaluating Vision-Language Models (VLMs) on pixel-level perception and reasoning tasks. It combines two complementary image regimes:

1. **Synthetic canvas** — controlled white-background images with precisely-sized geometric/semantic targets (letters, animals, shapes, blocks, dots).
2. **Text in the wild** (SynthText-style) — English words rendered onto real natural-scene photographs from the [SynthText](https://github.com/ankush-me/SynthText) `bg_img` set, with **pixel-accurate control of character cap-height**.

All images are **448 × 448 px**. The primary difficulty axis is the **target pixel size** (cap-height for text), swept over `[4, 8, 12, 16, 24, 32, 48]`, mapped to `extreme / hard / medium / easy`.

## Dataset Summary

| Split | #Samples | #Task types | Regimes |
|-------|---------:|:-----------:|---------|
| `perception` | 42 000 | 6 | synthetic canvas + text-in-the-wild |
| `reasoning`  | 39 200 | 6 | synthetic canvas + text-in-the-wild |

## Dataset Structure

### `perception` split — 42 000 samples

Single-target identification tasks. 7 000 samples per task, 1 000 samples per pixel size × 7 sizes.

| `task_type` | Description | Source |
|-------------|-------------|--------|
| `letter_recognition` | Identify a rendered uppercase letter (A–Z) | synthetic canvas |
| `animal_recognition` | Identify an animal silhouette (cat/dog/fish/bird/rabbit/turtle) | synthetic canvas |
| `shape_recognition` | Identify a geometric shape (circle/triangle/square/star/diamond/pentagon/hexagon/cross) | synthetic canvas |
| `block_recognition` | Detect / count square blocks | synthetic canvas |
| `color_block_recognition` | Identify the color of a block | synthetic canvas |
| `text_recognition` | Read a single English word overlaid on a natural scene | **text in the wild** |

### `reasoning` split — 39 200 samples

Chain-reasoning tasks requiring counting, ordering, and spatial reasoning across multiple targets.

| `task_type` | Description | Source |
|-------------|-------------|--------|
| `spatial_chain` | List all objects left→right or top→bottom | synthetic canvas |
| `comparison_chain` | List all objects smallest→largest by size | synthetic canvas |
| `counting_chain` | Count objects per type + total | synthetic canvas |
| `blur_chain` | Count objects on a blurred/textured background | synthetic canvas |
| `text_reading_chain` | Read multiple overlaid words in left→right / top→bottom order | **text in the wild** |
| `text_counting_chain` | Total word count + # words containing a queried letter | **text in the wild** |

### Difficulty levels

| Difficulty | Target / cap-height |
|------------|---------------------|
| `extreme`  | ≤ 5 px |
| `hard`     | 6–12 px |
| `medium`   | 13–24 px |
| `easy`     | 25–48 px |

## Fields

| Field | Type | Description |
|-------|------|-------------|
| `image`      | Image    | 448×448 PNG |
| `image_id`   | string   | Unique identifier (encodes task, size, count) |
| `task_type`  | string   | See tables above |
| `question`   | string   | Prompt for the VLM (asks for a structured JSON answer) |
| `answer`     | string   | Ground-truth answer. JSON-encoded (see below) |
| `difficulty` | string   | `easy` / `medium` / `hard` / `extreme` |
| `metadata`   | string   | JSON with canvas size, target pixel size, positions, colors, bounding boxes, sub-answers, etc. |

### Answer schemas (examples)

| Task | Answer JSON |
|------|-------------|
| `letter_recognition` | `{"letter": "H"}` |
| `animal_recognition` | `{"animal": "rabbit"}` |
| `shape_recognition`  | `{"shape": "triangle"}` |
| `color_block_recognition` | `{"color": "blue"}` |
| `text_recognition`   | `{"word": "HOME"}` |
| `spatial_chain`      | `{"objects": ["red A", "blue K", ...]}` |
| `comparison_chain`   | `{"objects": ["blue dog", "magenta bird"]}` |
| `counting_chain`     | `{"counts": {"red": 2, "blue": 1}, "total": 3}` |
| `blur_chain`         | `{"counts": {"circle": 1, "square": 2}, "total": 3}` |
| `text_reading_chain` | `{"words": ["HOME", "CITY", "EXIT"]}` |
| `text_counting_chain`| `{"total": 6, "with_letter": 3}` |

## Usage

```python
from datasets import load_dataset

ds = load_dataset("Volavion/FineSightBench-Large")
print(ds)
# DatasetDict({
#     perception: Dataset({features: [...], num_rows: 42000}),
#     reasoning:  Dataset({features: [...], num_rows: 39200})
# })

sample = ds["perception"][0]
sample["image"].show()
print(sample["question"])
print(sample["answer"])      # JSON string, e.g. '{"letter": "A"}'
```

Filter by task or difficulty:

```python
text_subset = ds["perception"].filter(lambda x: x["task_type"] == "text_recognition")
extreme    = ds["perception"].filter(lambda x: x["difficulty"] == "extreme")
```

## Design Philosophy

* **Pixel-size is the primary difficulty axis.** Targets (objects or characters) are rendered at exact cap-heights across `[4, 8, 12, 16, 24, 32, 48]` px so that the same semantic task can be probed from *easily readable* to *near-imperceptible* scales on a single fixed 448×448 canvas.
* **Controlled composition.** Every sample exposes pixel-precise target positions, bounding boxes, colors (with RGB), and sub-answers in `metadata`, enabling per-task, per-size, per-color, and positional analyses.
* **Two image regimes.** The synthetic canvas removes distribution confounders, while the SynthText-style text-in-the-wild regime stresses models with the same text task on varied, real photographs.

## Generation

Generated with the [FineSightBench repository](https://github.com/Volavion/FineSightBench):

```bash
# 10× base counts (perception: --num-per-config 1000, reasoning: N_PER_CONFIG=200)
python scripts/generate_large_dataset.py   # FSB_LARGE_SCALE=10 by default
```

**Text-in-the-wild backgrounds**: the first ~1 500 JPEGs from the SynthText `bg_img.tar.gz` set ([mirror](https://thor.robots.ox.ac.uk/scenetext/preproc/bg_img.tar.gz)) are center-cropped and resized to 448×448. Text glyphs use system sans-serif fonts; cap-height is calibrated per render to match the requested pixel size exactly.

## Citation

If you use FineSightBench, please cite the repository and the SynthText background source:

```bibtex
@misc{finesightbench_large2026,
  title  = {FineSightBench-Large: 10\times Scaled Fine-grained Visual Perception \& Reasoning Benchmark for VLMs},
  year   = {2026},
  url    = {https://huggingface.co/datasets/Volavion/FineSightBench-Large}
}

@inproceedings{Gupta16,
  author    = {A. Gupta and A. Vedaldi and A. Zisserman},
  title     = {Synthetic Data for Text Localisation in Natural Images},
  booktitle = {IEEE Conference on Computer Vision and Pattern Recognition},
  year      = {2016}
}
```

## License

Apache-2.0 for the FineSightBench benchmark code, annotations, and synthetic images. The natural-scene backgrounds for the text-in-the-wild tasks are derived from the SynthText `bg_img` set; please refer to the original SynthText dataset for the background-image license/terms.
"""

upload_file(
    path_or_fileobj=README.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    commit_message="Add dataset card for FineSightBench-Large v1.0",
)
print("README.md uploaded.")


## 8. Verify Upload

In [13]:
from datasets import load_dataset

ds_check = load_dataset(HF_REPO_ID)
print(ds_check)
print()
print("Perception sample 0:")
s = ds_check["perception"][0]
print(f"  image_id  : {s['image_id']}")
print(f"  task_type : {s['task_type']}")
print(f"  question  : {s['question'][:80]}...")
print(f"  answer    : {s['answer']}")
print(f"  difficulty: {s['difficulty']}")
print()
print("Reasoning sample 0:")
s = ds_check["reasoning"][0]
print(f"  image_id  : {s['image_id']}")
print(f"  task_type : {s['task_type']}")
print(f"  question  : {s['question'][:80]}...")
print(f"  answer    : {s['answer'][:80]}")
print(f"  difficulty: {s['difficulty']}")

Generating perception split: 3500 examples [00:00, 88705.04 examples/s]
Generating reasoning split: 2520 examples [00:00, 14734.13 examples/s]


DatasetDict({
    perception: Dataset({
        features: ['image', 'image_id', 'task_type', 'question', 'answer', 'difficulty', 'metadata'],
        num_rows: 3500
    })
    reasoning: Dataset({
        features: ['image', 'image_id', 'task_type', 'question', 'answer', 'difficulty', 'metadata'],
        num_rows: 2520
    })
})

Perception sample 0:
  image_id  : perception_letter_4px_00000
  task_type : letter_recognition
  question  : What letter is displayed in the image? Answer in JSON format only: {"letter": "<...
  answer    : {"letter": "A"}
  difficulty: extreme

Reasoning sample 0:
  image_id  : reasoning_spatial_chain_4px_n2_00000
  task_type : chain_reasoning
  question  : List all dots in the image from top to bottom. Describe each by its color and id...
  answer    : {"objects": ["yellow dot", "blue dot"]}
  difficulty: extreme
